In [1]:
# 第一个代码块：导入库和基础设置（更新版）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
import json
import logging
from pathlib import Path
import sys
from datetime import datetime
import yaml
warnings.filterwarnings('ignore')

# 设置中文字体和绘图样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# 添加项目路径
project_root = Path("/media/marxin/softs/workspace/Whisper_of_DNA_pl")
sys.path.append(str(project_root))
sys.path.append(str(project_root / "training"))

print("环境设置完成")

环境设置完成


In [ ]:
# 第二个代码块：配置和路径设置（针对您的模型）
# 配置参数
CONFIG = {
    # 模型相关
    "model_checkpoint_path": "/media/marxin/softs/workspace/Whisper_of_DNA_pl/output/checkpoints_wheatpca_599_envall_pretrain/run0_wheatpca_599_envall_30k/last.ckpt",
    "model_config_path": project_root / "config" / "model_config.json",  # 模型配置文件
    
    # 数据相关 - 根据您的实际数据路径修改
    "data_type": "hdf5",  # 或 "csv"，根据您使用的数据格式
    "h5_file_path": "path/to/your/test_data.h5",  # 替换为您的测试数据路径
    "genotype_csv_path": "path/to/test_genotype.csv",  # 如果使用CSV格式
    "phenotype_csv_path": "path/to/test_phenotype.csv",  # 如果使用CSV格式
    "training_config_path": project_root / "config" / "training_config.yml",
    
    # 分析参数
    "sample_size_shap": 300,      # SHAP分析样本数（减少以提高速度）
    "sample_size_tsne": 1000,     # t-SNE分析样本数
    "tsne_perplexity": 30,        # t-SNE困惑度
    "tsne_n_iter": 1000,          # t-SNE迭代次数
    "random_seed": 42,            # 随机种子
    
    # 特征提取层（可以选择不同层进行分析）
    "extract_from_layer": "gfi_output",  # "embedding", "gfi_output", "final_pred"
    
    # 输出路径
    "output_dir": project_root / "analysis_results" / "dna_whisper_analysis",
    "save_figures": True,
    "figure_format": "png",       # 图片格式
    "dpi": 300                    # 图片分辨率
}

# 创建输出目录
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

# 设置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CONFIG["output_dir"] / "analysis.log")
    ]
)
logger = logging.getLogger("DNAWhisperAnalysis")

print("配置设置完成")
print(f"模型路径: {CONFIG['model_checkpoint_path']}")
print(f"输出目录: {CONFIG['output_dir']}")

In [ ]:
# 第三个代码块：加载配置和数据模块（针对DNAWhisper）
def load_configs():
    """加载配置文件"""
    configs = {}
    
    # 尝试从检查点中加载配置
    try:
        checkpoint = torch.load(CONFIG["model_checkpoint_path"], map_location='cpu')
        if 'hyper_parameters' in checkpoint:
            configs["model_config"] = checkpoint['hyper_parameters'].get('model_config', {})
            configs["training_config"] = checkpoint['hyper_parameters'].get('config', {})
            logger.info("从检查点加载配置成功")
        else:
            logger.warning("检查点中未找到超参数，尝试加载外部配置文件")
            raise KeyError("No hyper_parameters in checkpoint")
    except Exception as e:
        logger.warning(f"从检查点加载配置失败: {e}")
        
        # 加载外部配置文件
        if CONFIG["model_config_path"].exists():
            with open(CONFIG["model_config_path"], 'r') as f:
                configs["model_config"] = json.load(f)
            logger.info("成功加载外部模型配置")
        else:
            # 创建默认模型配置
            configs["model_config"] = {
                "embedding": {
                    "input_type": "SNP",  # 根据您的数据类型调整
                    "input_dims": 10,     # 根据实际SNP特征维度调整
                },
                "output_layer": {
                    "phenotype_dim": 1,   # 根据实际表型数量调整
                    "phenotype_name": ["target_phenotype"]
                }
            }
            logger.warning("使用默认模型配置")
        
        # 加载训练配置
        if CONFIG["training_config_path"].exists():
            with open(CONFIG["training_config_path"], 'r') as f:
                configs["training_config"] = yaml.safe_load(f)
            logger.info("成功加载训练配置")
        else:
            configs["training_config"] = {
                "data": {
                    "batch_size": 16,
                    "num_workers": 4,
                    "normalize_phenotype": True,
                    "phenotype_norm_method": "minmax"
                }
            }
            logger.warning("使用默认训练配置")
    
    return configs

def setup_data_module(configs):
    """设置数据模块"""
    if CONFIG["data_type"] == "hdf5":
        from training.data.datamodule_onlySNP import WhisperDNADataModule_onlySNP
        
        # 获取表型名称
        phenotype_names = configs["model_config"].get("output_layer", {}).get("phenotype_name", ["target"])
        
        data_module = WhisperDNADataModule_onlySNP(
            h5_file_path=CONFIG["h5_file_path"],
            config=configs["training_config"],
            model_config=configs["model_config"],
            phenotype_names=phenotype_names,
            seed=CONFIG["random_seed"],
            logger=logger
        )
        
    elif CONFIG["data_type"] == "csv":
        from training.data.datamodule_bycsv import WhisperDNADataModule_byCSV
        
        # 更新训练配置以包含CSV路径
        configs["training_config"]["data"]["genotype_csv_path"] = str(CONFIG["genotype_csv_path"])
        configs["training_config"]["data"]["phenotype_csv_path"] = str(CONFIG["phenotype_csv_path"])
        
        phenotype_names = configs["model_config"].get("output_layer", {}).get("phenotype_name", ["target"])
        
        data_module = WhisperDNADataModule_byCSV(
            config=configs["training_config"],
            model_config=configs["model_config"],
            phenotype_names=phenotype_names,
            seed=CONFIG["random_seed"],
            logger=logger
        )
    else:
        raise ValueError(f"不支持的数据类型: {CONFIG['data_type']}")
    
    return data_module

# 加载配置和设置数据模块
configs = load_configs()
if configs is None:
    raise RuntimeError("配置加载失败")

# 如果没有实际数据文件，创建模拟数据用于演示
if not Path(CONFIG.get("h5_file_path", "")).exists() and not Path(CONFIG.get("genotype_csv_path", "")).exists():
    logger.warning("未找到实际数据文件，将创建模拟数据用于演示")
    # 这里可以创建模拟数据或者跳过数据模块初始化
    data_module = None
else:
    data_module = setup_data_module(configs)
    data_module.prepare_data()
    data_module.setup()
    
    logger.info("数据模块设置完成")
    print(f"数据集信息:")
    print(f"- 特征维度: {data_module.feature_dim}")
    print(f"- 表型数量: {data_module.num_phenotypes}")
    print(f"- 表型名称: {data_module.phenotype_names}")
    if hasattr(data_module, 'num_snps'):
        print(f"- SNP数量: {data_module.num_snps}")

print("配置和数据准备完成")

In [ ]:
# 第四个代码块：DNAWhisper模型加载器
class DNAWhisperAnalyzer:
    def __init__(self, model_path, model_config, data_module=None):
        """
        初始化DNAWhisper模型分析器
        
        Args:
            model_path: 模型检查点路径
            model_config: 模型配置字典
            data_module: 数据模块实例
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_config = model_config
        self.data_module = data_module
        self.model = None
        self.checkpoint = None
        
        # 加载模型
        if Path(model_path).exists():
            self.model, self.checkpoint = self.load_dna_whisper_model(model_path)
            if self.model is not None:
                logger.info(f"DNAWhisper模型加载成功，使用设备: {self.device}")
            else:
                logger.warning("模型加载失败，将创建演示模型")
                self.model = self.create_demo_model()
        else:
            logger.warning(f"模型文件不存在: {model_path}")
            self.model = self.create_demo_model()
        
        if self.model is not None:
            self.model.eval()
        
    def load_dna_whisper_model(self, model_path):
        """加载DNAWhisper模型"""
        try:
            # 导入模型类
            from training.models.model import DNAWhisperModel
            
            # 加载检查点
            checkpoint = torch.load(model_path, map_location=self.device)
            logger.info("检查点加载成功")
            
            # 获取模型配置
            if 'hyper_parameters' in checkpoint:
                model_config = checkpoint['hyper_parameters'].get('model_config', self.model_config)
            else:
                model_config = self.model_config
            
            # 创建模型实例
            model = DNAWhisperModel(config=model_config)
            
            # 加载权重
            if 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
                logger.info("从state_dict加载权重")
            else:
                state_dict = checkpoint
                logger.info("直接加载权重")
            
            # 移除Lightning模块前缀（如果存在）
            new_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith('model.'):
                    new_key = k[6:]  # 移除'model.'前缀
                else:
                    new_key = k
                new_state_dict[new_key] = v
            
            model.load_state_dict(new_state_dict, strict=False)
            model.to(self.device)
            
            logger.info("模型权重加载完成")
            
            # 打印模型信息
            total_params = sum(p.numel() for p in model.parameters())
            trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            logger.info(f"模型参数总数: {total_params:,}")
            logger.info(f"可训练参数: {trainable_params:,}")
            
            return model, checkpoint
            
        except Exception as e:
            logger.error(f"DNAWhisper模型加载失败: {e}")
            import traceback
            logger.error(traceback.format_exc())
            return None, None
    
    def create_demo_model(self):
        """创建演示模型"""
        try:
            from training.models.model import DNAWhisperModel
            
            # 创建最小配置
            demo_config = {
                "embedding": {
                    "input_type": "SNP",
                    "input_dims": 10,
                    "context_length": 100,
                    "num_blocks": 1,
                    "cnn_config": {
                        "enabled": True,
                        "num_layers": 2,
                        "channels": [32, 64],
                        "kernel_sizes": [3, 3],
                        "pooling": {"type": "average", "kernel_size": 2}
                    }
                },
                "GFI_FormerBLOCKS": {
                    "num_blocks": 1,
                    "blocks": [{
                        "name": "demo_block",
                        "context_length": 50,
                        "encoder": {
                            "num_layers": 1,
                            "attention": {"type": "standard", "num_heads": 4, "hidden_dims": 128}
                        },
                        "decoder": {
                            "cross_attention": {"type": "standard", "num_heads": 4},
                            "MOE": {"num_experts": 2, "experts_dims": 64},
                            "pooling": {"type": "average"}
                        }
                    }]
                },
                "output_layer": {
                    "phenotype_dim": 1,
                    "hidden_dims": [128, 64],
                    "activation": "ReLU",
                    "dropout_rate": 0.1
                }
            }
            
            model = DNAWhisperModel(config=demo_config)
            model.to(self.device)
            logger.info("创建了演示DNAWhisper模型")
            return model
            
        except Exception as e:
            logger.error(f"创建演示模型失败: {e}")
            return None
    
    def extract_features_from_layer(self, dataloader, layer_name="gfi_output"):
        """从指定层提取特征"""
        if self.model is None:
            logger.error("模型未加载")
            return None, None, None
        
        features_list = []
        labels_list = []
        predictions_list = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(dataloader):
                try:
                    # 处理批次数据
                    if isinstance(batch, dict):
                        batch_features = batch["features"].to(self.device)
                        batch_labels = batch["phenotype"].to(self.device)
                    elif isinstance(batch, (list, tuple)) and len(batch) >= 2:
                        batch_features = batch[0].to(self.device)
                        batch_labels = batch[1].to(self.device)
                    else:
                        logger.warning(f"跳过批次 {batch_idx}: 不支持的数据格式")
                        continue
                    
                    # 前向传播
                    outputs = self.model(batch_features)
                    
                    # 根据指定层提取特征
                    if layer_name == "embedding":
                        extracted_features = outputs['embed_features']
                    elif layer_name == "gfi_output":
                        if outputs['gfi_block_features']:
                            extracted_features = outputs['gfi_block_features'][-1]  # 最后一个GFI块的输出
                        else:
                            extracted_features = outputs['embed_features']
                    elif layer_name == "final_pred":
                        extracted_features = outputs['final_pred']
                    else:
                        logger.warning(f"未知的层名称: {layer_name}，使用final_pred")
                        extracted_features = outputs['final_pred']
                    
                    # 处理多维特征
                    if extracted_features.dim() > 2:
                        # 平均池化或展平
                        if extracted_features.dim() == 3:  # [batch, seq, dim]
                            extracted_features = extracted_features.mean(dim=1)
                        else:  # 更高维度，展平
                            extracted_features = extracted_features.reshape(extracted_features.size(0), -1)
                    
                    features_list.append(extracted_features.cpu().numpy())
                    labels_list.append(batch_labels.cpu().numpy())
                    predictions_list.append(outputs['final_pred'].cpu().numpy())
                    
                    # 限制批次数量以节省内存
                    if batch_idx >= 50:  # 最多处理50个批次
                        logger.info(f"已处理{batch_idx+1}个批次，停止以节省内存")
                        break
                        
                except Exception as e:
                    logger.error(f"处理批次 {batch_idx} 时出错: {e}")
                    continue
        
        if not features_list:
            logger.error("没有成功提取任何特征")
            return None, None, None
        
        features = np.vstack(features_list)
        labels = np.vstack(labels_list)
        predictions = np.vstack(predictions_list)
        
        logger.info(f"特征提取完成: {features.shape}, 标签: {labels.shape}")
        
        return features, labels, predictions

# 创建DNAWhisper分析器
analyzer = DNAWhisperAnalyzer(
    model_path=CONFIG["model_checkpoint_path"],
    model_config=configs["model_config"],
    data_module=data_module
)

print("DNAWhisper模型分析器创建完成")

In [ ]:
# 第五个代码块：创建模拟数据（如果没有实际数据）
def create_mock_data_for_analysis(config, analyzer):
    """创建模拟数据用于分析演示"""
    logger.info("创建模拟数据用于分析演示...")
    
    # 模拟参数
    n_samples = config["sample_size_tsne"]
    n_snps = 1000
    snp_dim = 10  # 每个SNP的特征维度
    n_phenotypes = 1
    
    # 生成模拟SNP数据
    np.random.seed(config["random_seed"])
    
    # 创建模拟基因型数据 [n_samples, n_snps, snp_dim]
    mock_genotype = np.random.randint(0, 3, size=(n_samples, n_snps, snp_dim)).astype(np.float32)
    
    # 添加一些结构化模式
    for i in range(0, n_snps, 100):
        end_idx = min(i + 50, n_snps)
        mock_genotype[:n_samples//3, i:end_idx, :] += 0.5  # 第一组样本的特定区域
        mock_genotype[n_samples//3:2*n_samples//3, i:end_idx, :] -= 0.3  # 第二组样本
    
    # 创建模拟表型数据
    # 基于前几个SNP创建相关的表型
    phenotype_signal = mock_genotype[:, :10, 0].mean(axis=1)  # 使用前10个SNP的平均值
    noise = np.random.normal(0, 0.1, n_samples)
    mock_phenotype = phenotype_signal + noise
    
    # 创建类别标签（用于分类可视化）
    mock_labels = np.zeros(n_samples, dtype=int)
    mock_labels[n_samples//3:2*n_samples//3] = 1
    mock_labels[2*n_samples//3:] = 2
    
    # 如果有模型，尝试提取特征
    if analyzer.model is not None:
        try:
            # 创建mock tensor
            mock_tensor = torch.FloatTensor(mock_genotype).to(analyzer.device)
            
            with torch.no_grad():
                outputs = analyzer.model(mock_tensor)
                
                # 提取不同层的特征
                if config["extract_from_layer"] == "embedding":
                    extracted_features = outputs['embed_features']
                elif config["extract_from_layer"] == "gfi_output":
                    if outputs['gfi_block_features']:
                        extracted_features = outputs['gfi_block_features'][-1]
                    else:
                        extracted_features = outputs['embed_features']
                else:
                    extracted_features = outputs['final_pred']
                
                # 处理特征维度
                if extracted_features.dim() > 2:
                    if extracted_features.dim() == 3:
                        extracted_features = extracted_features.mean(dim=1)
                    else:
                        extracted_features = extracted_features.reshape(extracted_features.size(0), -1)
                
                features = extracted_features.cpu().numpy()
                predictions = outputs['final_pred'].cpu().numpy()
                
                logger.info(f"使用模型提取的特征维度: {features.shape}")
                
        except Exception as e:
            logger.error(f"模型特征提取失败: {e}")
            # 使用原始特征
            features = mock_genotype.reshape(n_samples, -1)
            predictions = mock_phenotype.reshape(-1, 1)
    else:
        # 直接使用展平的基因型数据
        features = mock_genotype.reshape(n_samples, -1)
        predictions = mock_phenotype.reshape(-1, 1)
    
    # 创建特征名称
    feature_names = []
    if analyzer.model is not None and features.shape[1] < 5000:
        feature_names = [f"ModelFeature_{i}" for i in range(features.shape[1])]
    else:
        for snp_idx in range(min(n_snps, features.shape[1]//snp_dim)):
            for dim_idx in range(snp_dim):
                feature_names.append(f"SNP_{snp_idx}_dim_{dim_idx}")
    
    # 补充特征名称（如果不够）
    while len(feature_names) < features.shape[1]:
        feature_names.append(f"Feature_{len(feature_names)}")
    
    mock_data = {
        'features_shap': features[:config["sample_size_shap"]],
        'labels_shap': mock_labels[:config["sample_size_shap"]],
        'features_tsne': features,
        'labels_tsne': mock_labels,
        'feature_names': feature_names,
        'is_classification': True,
        'phenotype_name': 'mock_phenotype',
        'all_features': features,
        'all_labels': mock_labels,
        'predictions': predictions,
        'mock_genotype': mock_genotype,
        'mock_phenotype_continuous': mock_phenotype
    }
    
    logger.info(f"模拟数据创建完成:")
    logger.info(f"- 样本数: {n_samples}")
    logger.info(f"- 特征维度: {features.shape[1]}")
    logger.info(f"- 类别数: {len(np.unique(mock_labels))}")
    
    return mock_data

def prepare_real_data(data_module, analyzer, config):
    """准备真实数据"""
    logger.info("准备真实数据进行分析...")
    
    # 获取测试数据
    test_dataloader = data_module.test_dataloader()
    if len(test_dataloader.dataset) == 0:
        test_dataloader = data_module.val_dataloader()
    if len(test_dataloader.dataset) == 0:
        test_dataloader = data_module.train_dataloader()
    
    # 提取特征
    features, labels, predictions = analyzer.extract_features_from_layer(
        test_dataloader, 
        config["extract_from_layer"]
    )
    
    if features is None:
        return None
    
    # 处理标签
    if labels.ndim > 1 and labels.shape[1] > 1:
        labels_for_viz = labels[:, 0]
    else:
        labels_for_viz = labels.ravel()
    
    # 判断是否为分类问题
    unique_labels = np.unique(labels_for_viz)
    is_classification = len(unique_labels) < 20 and np.allclose(labels_for_viz, labels_for_viz.astype(int))
    
    # 采样
    n_samples = len(features)
    sample_size_shap = min(config["sample_size_shap"], n_samples)
    sample_size_tsne = min(config["sample_size_tsne"], n_samples)
    
    indices = np.random.permutation(n_samples)
    shap_indices = indices[:sample_size_shap]
    tsne_indices = indices[:sample_size_tsne]
    
    # 创建特征名称
    if hasattr(data_module, 'num_snps') and data_module.num_snps:
        feature_names = [f"Feature_{i}" for i in range(features.shape[1])]
    else:
        feature_names = [f"Feature_{i}" for i in range(features.shape[1])]
    
    real_data = {
        'features_shap': features[shap_indices],
        'labels_shap': labels_for_viz[shap_indices],
        'features_tsne': features[tsne_indices],
        'labels_tsne': labels_for_viz[tsne_indices],
        'feature_names': feature_names,
        'is_classification': is_classification,
        'phenotype_name': data_module.phenotype_names[0] if data_module.phenotype_names else 'phenotype',
        'all_features': features,
        'all_labels': labels_for_viz,
        'predictions': predictions
    }
    
    return real_data

# 准备分析数据
if data_module is not None:
    try:
        analysis_data = prepare_real_data(data_module, analyzer, CONFIG)
        if analysis_data is None:
            logger.warning("真实数据准备失败，使用模拟数据")
            analysis_data = create_mock_data_for_analysis(CONFIG, analyzer)
    except Exception as e:
        logger.error(f"真实数据准备出错: {e}")
        analysis_data = create_mock_data_for_analysis(CONFIG, analyzer)
else:
    analysis_data = create_mock_data_for_analysis(CONFIG, analyzer)

print("分析数据准备完成")
print(f"- SHAP分析样本数: {len(analysis_data['features_shap'])}")
print(f"- t-SNE分析样本数: {len(analysis_data['features_tsne'])}")
print(f"- 特征维度: {analysis_data['features_shap'].shape[1]}")
print(f"- 问题类型: {'分类' if analysis_data['is_classification'] else '回归'}")

In [ ]:
# 第六个代码块：SHAP分析（针对DNAWhisper模型）
def perform_dna_whisper_shap_analysis(analyzer, features, labels, feature_names, config):
    """对DNAWhisper模型执行SHAP分析"""
    logger.info("开始DNAWhisper模型的SHAP分析...")
    
    if analyzer.model is None:
        logger.warning("模型为空，跳过SHAP分析")
        return None, None, None
    
    try:
        # 准备模型预测函数
        def model_predict_wrapper(x):
            """包装模型预测函数供SHAP使用"""
            if isinstance(x, np.ndarray):
                x = torch.FloatTensor(x)
            
            analyzer.model.eval()
            with torch.no_grad():
                device = next(analyzer.model.parameters()).device
                x = x.to(device)
                
                # 根据模型输入格式调整
                if x.dim() == 2:  # [batch, features]
                    # 如果是展平的特征，需要重塑为模型期望的格式
                    if hasattr(analysis_data, 'mock_genotype'):
                        # 对于模拟数据，重塑为 [batch, n_snps, snp_dim]
                        original_shape = analysis_data['mock_genotype'].shape
                        n_snps, snp_dim = original_shape[1], original_shape[2]
                        if x.shape[1] == n_snps * snp_dim:
                            x = x.reshape(x.shape[0], n_snps, snp_dim)
                
                outputs = analyzer.model(x)
                
                # 返回最终预测
                final_pred = outputs['final_pred']
                
                # 如果是多输出，只返回第一个
                if final_pred.dim() > 1 and final_pred.shape[1] > 1:
                    final_pred = final_pred[:, 0:1]
                
                return final_pred.cpu().numpy()
        
        # 选择背景数据集
        background_size = min(50, len(features))  # 减少背景样本数以节省内存
        background = features[:background_size]
        
        # 选择要解释的样本
        explain_size = min(100, len(features))  # 减少解释样本数
        X_explain = features[:explain_size]
        
        logger.info(f"使用{background_size}个样本作为背景，解释{explain_size}个样本")
        
        # 创建SHAP explainer
        # 对于复杂的深度学习模型，使用KernelExplainer
        logger.info("创建SHAP KernelExplainer...")
        explainer = shap.KernelExplainer(model_predict_wrapper, background)
        
        # 计算SHAP值
        logger.info("计算SHAP值中...（这可能需要几分钟）")
        shap_values = explainer.shap_values(X_explain, silent=True)
        
        # 如果shap_values是列表（多输出），取第一个
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
        
        logger.info(f"SHAP值计算完成，形状: {shap_values.shape}")
        
        return explainer, shap_values, X_explain
        
    except Exception as e:
        logger.error(f"SHAP分析失败: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return None, None, None

def plot_dna_whisper_shap_visualizations(explainer, shap_values, X_explain, feature_names, config):
    """绘制DNAWhisper模型的SHAP可视化图表"""
    if explainer is None or shap_values is None:
        logger.warning("SHAP数据为空，跳过可视化")
        return
    
    output_dir = config["output_dir"]
    
    try:
        # 1. 特征重要性分析
        feature_importance = np.mean(np.abs(shap_values), axis=0)
        
        # Top重要特征
        max_features = min(20, len(feature_importance))
        top_features_idx = np.argsort(feature_importance)[-max_features:]
        
        # SHAP Summary Plot
        plt.figure(figsize=(12, 8))
        try:
            shap.summary_plot(
                shap_values[:, top_features_idx], 
                X_explain[:, top_features_idx],
                feature_names=[feature_names[i] if i < len(feature_names) else f"Feature_{i}" 
                              for i in top_features_idx],
                show=False
            )
            plt.title('DNAWhisper模型 - SHAP特征重要性总览', fontsize=14, pad=20)
            plt.tight_layout()
            
            if config["save_figures"]:
                plt.savefig(output_dir / f"dna_whisper_shap_summary.{config['figure_format']}", 
                           dpi=config["dpi"], bbox_inches='tight')
            plt.show()
        except Exception as e:
            logger.error(f"SHAP summary plot失败: {e}")
        
        # 2. SHAP Bar Plot
        plt.figure(figsize=(10, 6))
        try:
            shap.summary_plot(
                shap_values[:, top_features_idx], 
                X_explain[:, top_features_idx],
                plot_type="bar",
                feature_names=[feature_names[i] if i < len(feature_names) else f"Feature_{i}" 
                              for i in top_features_idx],
                show=False
            )
            plt.title('DNAWhisper模型 - 平均特征重要性', fontsize=14, pad=20)
            plt.tight_layout()
            
            if config["save_figures"]:
                plt.savefig(output_dir / f"dna_whisper_shap_bar.{config['figure_format']}", 
                           dpi=config["dpi"], bbox_inches='tight')
            plt.show()
        except Exception as e:
            logger.error(f"SHAP bar plot失败: {e}")
        
        # 3. 单样本解释（Waterfall plot）
        if len(shap_values) > 0:
            plt.figure(figsize=(12, 8))
            try:
                # 选择SHAP值绝对值最大的样本
                sample_shap_magnitude = np.sum(np.abs(shap_values), axis=1)
                most_important_sample_idx = np.argmax(sample_shap_magnitude)
                
                shap.waterfall_plot(
                    explainer.expected_value, 
                    shap_values[most_important_sample_idx, top_features_idx], 
                    X_explain[most_important_sample_idx, top_features_idx],
                    feature_names=[feature_names[i] if i < len(feature_names) else f"Feature_{i}" 
                                  for i in top_features_idx],
                    show=False
                )
                plt.title(f'DNAWhisper模型 - 样本{most_important_sample_idx}的特征贡献', fontsize=14)
                plt.tight_layout()
                
                if config["save_figures"]:
                    plt.savefig(output_dir / f"dna_whisper_shap_waterfall.{config['figure_format']}", 
                               dpi=config["dpi"], bbox_inches='tight')
                plt.show()
            except Exception as e:
                logger.error(f"SHAP waterfall plot失败: {e}")
        
        # 4. 特征重要性统计
        importance_df = pd.DataFrame({
            'feature_idx': range(len(feature_importance)),
            'feature_name': [feature_names[i] if i < len(feature_names) else f"Feature_{i}" 
                            for i in range(len(feature_importance))],
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print("\nDNAWhisper模型 - Top 15 最重要特征:")
        print(importance_df.head(15).to_string(index=False))
        
        # 保存特征重要性
        if config["save_figures"]:
            importance_df.to_csv(output_dir / "dna_whisper_feature_importance.csv", index=False)
        
        # 5. SHAP值分布分析
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.hist(shap_values.flatten(), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        plt.xlabel('SHAP值')
        plt.ylabel('频次')
        plt.title('SHAP值分布')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.hist(feature_importance, bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
        plt.xlabel('平均绝对SHAP值')
        plt.ylabel('特征数量')
        plt.title('特征重要性分布')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        if config["save_figures"]:
            plt.savefig(output_dir / f"dna_whisper_shap_distributions.{config['figure_format']}", 
                       dpi=config["dpi"], bbox_inches='tight')
        plt.show()
        
        logger.info("DNAWhisper SHAP可视化完成")
        
        return feature_importance
        
    except Exception as e:
        logger.error(f"SHAP可视化失败: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return None

# 执行SHAP分析
if analyzer.model is not None:
    explainer, shap_values, X_explain = perform_dna_whisper_shap_analysis(
        analyzer,
        analysis_data['features_shap'],
        analysis_data['labels_shap'],
        analysis_data['feature_names'],
        CONFIG
    )
    
    if shap_values is not None:
        feature_importance = plot_dna_whisper_shap_visualizations(
            explainer, shap_values, X_explain, 
            analysis_data['feature_names'], CONFIG
        )
    else:
        feature_importance = None
else:
    logger.warning("跳过SHAP分析（模型未加载）")
    explainer, shap_values, X_explain, feature_importance = None, None, None, None

print("DNAWhisper SHAP分析完成")

In [ ]:
# 第七个代码块：t-SNE分析和综合报告
def perform_tsne_analysis_for_dna_whisper(features, labels, config):
    """为DNAWhisper模型执行t-SNE分析"""
    logger.info("开始DNAWhisper模型的t-SNE分析...")
    
    try:
        # 标准化特征
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # 如果特征维度太高，先进行PCA降维
        if features_scaled.shape[1] > 50:
            from sklearn.decomposition import PCA
            n_components = min(50, features_scaled.shape[0] - 1)
            logger.info(f"特征维度过高({features_scaled.shape[1]})，使用PCA降维到{n_components}维")
            pca = PCA(n_components=n_components, random_state=config["random_seed"])
            features_scaled = pca.fit_transform(features_scaled)
            logger.info(f"PCA解释方差比例: {pca.explained_variance_ratio_.sum():.3f}")
        
        # 执行t-SNE
        logger.info("执行t-SNE降维...")
        perplexity = min(config["tsne_perplexity"], len(features) - 1, 30)
        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            n_iter=config["tsne_n_iter"],
            random_state=config["random_seed"],
            verbose=1
        )
        
        features_tsne = tsne.fit_transform(features_scaled)
        
        logger.info("DNAWhisper t-SNE分析完成")
        return features_tsne, scaler
        
    except Exception as e:
        logger.error(f"t-SNE分析失败: {e}")
        return None, None

def create_comprehensive_dna_whisper_report(analysis_data, shap_values, features_tsne, feature_importance, config):
    """生成DNAWhisper模型的综合分析报告"""
    logger.info("生成DNAWhisper模型综合分析报告...")
    
    output_dir = config["output_dir"]
    
    try:
        # 创建综合分析图
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. 模型架构信息
        axes[0, 0].axis('off')
        model_info = f"""
DNAWhisper模型分析报告

模型信息:
• 检查点: {Path(config['model_checkpoint_path']).name}
• 提取层: {config['extract_from_layer']}
• 分析时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

数据集统计:
• 总样本数: {len(analysis_data['all_features'])}
• 特征维度: {analysis_data['all_features'].shape[1]}
• 表型: {analysis_data['phenotype_name']}
• 问题类型: {'分类' if analysis_data['is_classification'] else '回归'}

分析参数:
• SHAP样本数: {len(analysis_data['features_shap'])}
• t-SNE样本数: {len(analysis_data['features_tsne'])}
• t-SNE困惑度: {config['tsne_perplexity']}
        """
        
        if analysis_data['is_classification']:
            unique_labels, counts = np.unique(analysis_data['labels_tsne'], return_counts=True)
            model_info += f"\n类别分布:\n"
            for label, count in zip(unique_labels, counts):
                model_info += f"• 类别 {int(label)}: {count} 样本\n"
        
        axes[0, 0].text(0.05, 0.95, model_info, transform=axes[0, 0].transAxes, 
                       fontsize=9, verticalalignment='top', fontfamily='monospace')
        
        # 2. t-SNE可视化
        if features_tsne is not None:
            if analysis_data['is_classification']:
                unique_labels = np.unique(analysis_data['labels_tsne'])
                colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = analysis_data['labels_tsne'] == label
                    axes[0, 1].scatter(features_tsne[mask, 0], features_tsne[mask, 1], 
                                      c=[colors[i]], label=f'类别 {int(label)}', alpha=0.7, s=20)
                axes[0, 1].legend(fontsize=8)
            else:
                scatter = axes[0, 1].scatter(features_tsne[:, 0], features_tsne[:, 1], 
                                           c=analysis_data['labels_tsne'], cmap='viridis', alpha=0.7, s=20)
                plt.colorbar(scatter, ax=axes[0, 1])
            
            axes[0, 1].set_xlabel('t-SNE 维度 1')
            axes[0, 1].set_ylabel('t-SNE 维度 2')
            axes[0, 1].set_title('DNAWhisper特征空间t-SNE可视化', fontsize=10)
            axes[0, 1].grid(True, alpha=0.3)
        else:
            axes[0, 1].text(0.5, 0.5, 't-SNE分析失败', ha='center', va='center', transform=axes[0, 1].transAxes)
        
        # 3. SHAP特征重要性
        if feature_importance is not None:
            top_10_indices = np.argsort(feature_importance)[-10:]
            top_10_importance = feature_importance[top_10_indices]
            top_10_names = [analysis_data['feature_names'][i] if i < len(analysis_data['feature_names']) 
                           else f'Feature_{i}' for i in top_10_indices]
            
            axes[0, 2].barh(range(10), top_10_importance)
            axes[0, 2].set_yticks(range(10))
            axes[0, 2].set_yticklabels([name[:15] + '...' if len(name) > 15 else name for name in top_10_names])
            axes[0, 2].set_title('Top 10 重要特征 (SHAP)', fontsize=10)
            axes[0, 2].set_xlabel('平均绝对SHAP值')
        else:
            axes[0, 2].text(0.5, 0.5, 'SHAP分析失败', ha='center', va='center', transform=axes[0, 2].transAxes)
        
        # 4. 预测vs真实值
        if 'predictions' in analysis_data and analysis_data['predictions'] is not None:
            predictions = analysis_data['predictions']
            if predictions.ndim > 1:
                predictions = predictions[:, 0]
            
            if analysis_data['is_classification']:
                # 分类问题：显示预测概率分布
                axes[1, 0].hist(predictions, bins=30, alpha=0.7, color='lightblue', edgecolor='black')
                axes[1, 0].set_xlabel('预测值')
                axes[1, 0].set_ylabel('频次')
                axes[1, 0].set_title('预测值分布', fontsize=10)
            else:
                # 回归问题：预测vs真实散点图
                axes[1, 0].scatter(analysis_data['all_labels'], predictions, alpha=0.5)
                axes[1, 0].plot([analysis_data['all_labels'].min(), analysis_data['all_labels'].max()], 
                               [analysis_data['all_labels'].min(), analysis_data['all_labels'].max()], 'r--')
                axes[1, 0].set_xlabel('真实值')
                axes[1, 0].set_ylabel('预测值')
                axes[1, 0].set_title('预测vs真实值', fontsize=10)
            axes[1, 0].grid(True, alpha=0.3)
        else:
            axes[1, 0].text(0.5, 0.5, '无预测数据', ha='center', va='center', transform=axes[1, 0].transAxes)
        
        # 5. 特征重要性分布
        if feature_importance is not None:
            axes[1, 1].hist(feature_importance, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
            axes[1, 1].set_xlabel('平均绝对SHAP值')
            axes[1, 1].set_ylabel('特征数量')
            axes[1, 1].set_title('特征重要性分布', fontsize=10)
            axes[1, 1].grid(True, alpha=0.3)
            
            # 添加统计信息
            stats_text = f"""
重要特征统计:
• 非零重要性: {np.sum(feature_importance > 0)}
• 高重要性(>平均值): {np.sum(feature_importance > np.mean(feature_importance))}
• 最大重要性: {np.max(feature_importance):.4f}
• 平均重要性: {np.mean(feature_importance):.4f}
            """
            axes[1, 1].text(0.02, 0.98, stats_text, transform=axes[1, 1].transAxes, 
                           fontsize=8, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
        else:
            axes[1, 1].text(0.5, 0.5, 'SHAP分析失败', ha='center', va='center', transform=axes[1, 1].transAxes)
        
        # 6. 类别分布（如果是分类问题）
        if analysis_data['is_classification']:
            unique_labels, counts = np.unique(analysis_data['all_labels'], return_counts=True)
            axes[1, 2].pie(counts, labels=[f'类别 {int(label)}' for label in unique_labels], 
                          autopct='%1.1f%%', startangle=90)
            axes[1, 2].set_title('类别分布', fontsize=10)
        else:
            # 回归问题：显示标签分布
            axes[1, 2].hist(analysis_data['all_labels'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
            axes[1, 2].set_xlabel('标签值')
            axes[1, 2].set_ylabel('频次')
            axes[1, 2].set_title('标签分布', fontsize=10)
            axes[1, 2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if config["save_figures"]:
            plt.savefig(output_dir / f"dna_whisper_comprehensive_analysis.{config['figure_format']}", 
                       dpi=config["dpi"], bbox_inches='tight')
        plt.show()
        
        # 生成详细文本报告
        report_path = output_dir / "dna_whisper_analysis_report.txt"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("=== DNAWhisper模型可解释性分析报告 ===\n\n")
            f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"模型检查点: {config['model_checkpoint_path']}\n")
            f.write(f"特征提取层: {config['extract_from_layer']}\n")
            f.write(f"分析表型: {analysis_data['phenotype_name']}\n")
            f.write(f"问题类型: {'分类' if analysis_data['is_classification'] else '回归'}\n\n")
            
            f.write("=== 数据集信息 ===\n")
            f.write(f"总样本数: {len(analysis_data['all_features'])}\n")
            f.write(f"特征维度: {analysis_data['all_features'].shape[1]}\n")
            f.write(f"SHAP分析样本数: {len(analysis_data['features_shap'])}\n")
            f.write(f"t-SNE分析样本数: {len(analysis_data['features_tsne'])}\n")
            
            if analysis_data['is_classification']:
                unique_labels, counts = np.unique(analysis_data['all_labels'], return_counts=True)
                f.write(f"类别数: {len(unique_labels)}\n")
                f.write("类别分布:\n")
                for label, count in zip(unique_labels, counts):
                    f.write(f"  类别 {int(label)}: {count} 样本 ({count/len(analysis_data['all_labels']):.2%})\n")
            
            if feature_importance is not None:
                f.write("\n=== SHAP特征重要性分析 ===\n")
                f.write(f"平均绝对SHAP值: {np.mean(feature_importance):.4f}\n")
                f.write(f"最大SHAP值: {np.max(feature_importance):.4f}\n")
                f.write(f"非零重要性特征数: {np.sum(feature_importance > 0)}\n")
                f.write(f"高重要性特征数(>平均值): {np.sum(feature_importance > np.mean(feature_importance))}\n")
                
                f.write("\nTop 20 重要特征:\n")
                top_20_indices = np.argsort(feature_importance)[-20:][::-1]
                for i, idx in enumerate(top_20_indices):
                    feature_name = analysis_data['feature_names'][idx] if idx < len(analysis_data['feature_names']) else f'Feature_{idx}'
                    f.write(f"{i+1:2d}. {feature_name}: {feature_importance[idx]:.4f}\n")
            
            if features_tsne is not None:
                f.write("\n=== t-SNE降维分析 ===\n")
                f.write(f"分析样本数: {len(features_tsne)}\n")
                f.write(f"困惑度: {config['tsne_perplexity']}\n")
                f.write(f"迭代次数: {config['tsne_n_iter']}\n")
                f.write(f"第一维度范围: [{features_tsne[:, 0].min():.2f}, {features_tsne[:, 0].max():.2f}]\n")
                f.write(f"第二维度范围: [{features_tsne[:, 1].min():.2f}, {features_tsne[:, 1].max():.2f}]\n")
        
        logger.info(f"DNAWhisper综合分析报告已保存到: {report_path}")
        
    except Exception as e:
        logger.error(f"生成综合报告失败: {e}")
        import traceback
        logger.error(traceback.format_exc())

# 执行t-SNE分析
features_tsne, scaler = perform_tsne_analysis_for_dna_whisper(
    analysis_data['features_tsne'],
    analysis_data['labels_tsne'],
    CONFIG
)

# 绘制t-SNE可视化
if features_tsne is not None:
    from training.analysis.analysis_flow import plot_tsne_visualizations  # 使用之前定义的函数
    try:
        plot_tsne_visualizations(
            features_tsne, 
            analysis_data['labels_tsne'],
            analysis_data['phenotype_name'],
            analysis_data['is_classification'],
            CONFIG
        )
    except:
        # 如果导入失败，使用简单的可视化
        plt.figure(figsize=(10, 8))
        if analysis_data['is_classification']:
            unique_labels = np.unique(analysis_data['labels_tsne'])
            colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
            for i, label in enumerate(unique_labels):
                mask = analysis_data['labels_tsne'] == label
                plt.scatter(features_tsne[mask, 0], features_tsne[mask, 1], 
                           c=[colors[i]], label=f'类别 {int(label)}', alpha=0.7)
            plt.legend()
        else:
            plt.scatter(features_tsne[:, 0], features_tsne[:, 1], 
                       c=analysis_data['labels_tsne'], cmap='viridis', alpha=0.7)
            plt.colorbar()
        
        plt.xlabel('t-SNE 维度 1')
        plt.ylabel('t-SNE 维度 2')
        plt.title(f'DNAWhisper模型 - t-SNE可视化 ({analysis_data["phenotype_name"]})')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if CONFIG["save_figures"]:
            plt.savefig(CONFIG["output_dir"] / f"dna_whisper_tsne_simple.{CONFIG['figure_format']}", 
                       dpi=CONFIG["dpi"], bbox_inches='tight')
        plt.show()

# 生成综合报告
create_comprehensive_dna_whisper_report(
    analysis_data, shap_values, features_tsne, feature_importance, CONFIG
)

print("\n=== DNAWhisper模型分析完成 ===")
print(f"结果保存在: {CONFIG['output_dir']}")
print("生成的文件包括:")
print("- dna_whisper_shap_*.png: SHAP分析图表")
print("- dna_whisper_tsne_*.png: t-SNE可视化图表") 
print("- dna_whisper_comprehensive_analysis.png: 综合分析图表")
print("- dna_whisper_analysis_report.txt: 详细分析报告")
print("- dna_whisper_feature_importance.csv: 特征重要性排序")